In [ ]:
import numpy as np
import pandas as pd
# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
import time
# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


In [3]:
#ML models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

In [4]:
# Cross Validation
from sklearn.model_selection import cross_validate, StratifiedKFold


In [5]:
#Evaluation metrics
from sklearn.metrics import (accuracy_score,precision_score, 
                             recall_score, f1_score,roc_auc_score, make_scorer,
                            confusion_matrix, classification_report,
                            average_precision_score)


In [6]:
#Display Themes 
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [7]:
train_df=pd.read_csv("UNSW_NB15_training-set.csv")

In [8]:
train_df.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,252,254,14158.942380,8495.365234,0,0,24.295600,8.375000,30.177547,11.830604,255,621772692,2202533631,255,0.000000,0.000000,0.000000,43,43,0,0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,62,252,8395.112305,503571.312500,2,17,49.915000,15.432865,61.426934,1387.778330,255,1417884146,3077387971,255,0.000000,0.000000,0.000000,52,1106,0,0,43,1,1,1,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,62,252,1572.271851,60929.230470,1,6,231.875571,102.737203,17179.586860,11420.926230,255,2116150707,2963114973,255,0.111897,0.061458,0.050439,46,824,0,0,7,1,2,1,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,62,252,2740.178955,3358.622070,1,3,152.876547,90.235726,259.080172,4991.784669,255,1107119177,1047442890,255,0.000000,0.000000,0.000000,52,64,0,0,1,1,2,1,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,254,252,8561.499023,3987.059814,2,1,47.750333,75.659602,2415.837634,115.807000,255,2436137549,1977154190,255,0.128381,0.071147,0.057234,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0


In [9]:
train_df.isnull().sum()

id                   0
dur                  0
proto                0
service              0
state                0
spkts                0
dpkts                0
sbytes               0
dbytes               0
rate                 0
sttl                 0
dttl                 0
sload                0
dload                0
sloss                0
dloss                0
sinpkt               0
dinpkt               0
sjit                 0
djit                 0
swin                 0
stcpb                0
dtcpb                0
dwin                 0
tcprtt               0
synack               0
ackdat               0
smean                0
dmean                0
trans_depth          0
response_body_len    0
ct_srv_src           0
ct_state_ttl         0
ct_dst_ltm           0
ct_src_dport_ltm     0
ct_dst_sport_ltm     0
ct_dst_src_ltm       0
is_ftp_login         0
ct_ftp_cmd           0
ct_flw_http_mthd     0
ct_src_ltm           0
ct_srv_dst           0
is_sm_ips_ports      0
attack_cat 

In [10]:
train_df.shape

(175341, 45)

In [11]:
unselected_col = ["id", "attack_cat", "label"]

X = train_df.drop(columns=unselected_col)
y = train_df["label"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (175341, 42)
Target shape: (175341,)


In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()
print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))

Numerical features: 39
Categorical features: 3


C:\Users\mssai\AppData\Local\Temp\ipykernel_2432\4206238514.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
#This pipelines for logistic regression model, we will use StandardScaler to scale the numerical features
#  and SimpleImputer to fill missing values with median.
numerical_pipeline_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
#This pipelines for decision tree model, we will use SimpleImputer to fill missing values with median.
#Tress don't require feature scaling, so we don't need to use StandardScaler here.
numerical_pipeline_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
#Categorical pipeline for both models, we will use SimpleImputer to fill missing 
# values with most frequent value and OneHotEncoder to encode categorical features.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [18]:
#Combining the Pipelines 
preprocessor_lr = ColumnTransformer([
    ("num", numerical_pipeline_lr, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

preprocessor_tree = ColumnTransformer([
    ("num", numerical_pipeline_tree, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

### Define the models

In [22]:
models={
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100,n_jobs=-1),
    "XGBOOST": XGBClassifier(random_state=42, n_estimators=100,n_jobs=2,
                              eval_metric='logloss')
}

## lets combine a pipelines with models

In [23]:
models_pipeline = {
    "Logistic Regression": Pipeline([
        ("preprocessing",preprocessor_lr),
        ("classifier",models["Logistic Regression"])
    ]),
    "DecisionTree": Pipeline([
        ("preprocessing", preprocessor_tree),
        ("classifier", models["DecisionTree"])
    ]),
    "Random Forest": Pipeline([
        ("preprocessing", preprocessor_tree),
        ("classifier", models["Random Forest"])
    ]),
    "XGBOOST": Pipeline([
        ("preprocessing", preprocessor_tree),
        ("classifier", models["XGBOOST"])
    ])  
}

In [24]:
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [26]:
scoring={
    "accuracy":"accuracy",
    "precesion":"precision",
    "recall":"recall",
    "f1":"f1",
    "roc_auc":"roc_auc",
    "average_precision":"average_precision"
}